In [218]:
import pandas as pd
import numpy as np

In [219]:
# first row contains 'Report Date'
df = pd.read_csv('../data/UK-Sanctions-List.csv', skiprows=1)

C:\Users\44743\AppData\Local\Temp\ipykernel_11204\3316590353.py:2: DtypeWarning: Columns (0: IMO number, 1: Current owner/operator (s), 2: Previous owner/operator (s), 3: Current believed flag of ship, 4: Previous flags, 5: Type of ship) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('../data/UK-Sanctions-List.csv', skiprows=1)


In [ ]:
# standardize column names
df.columns = (df.columns.
                str.replace(' ', '_').
                str.replace('-', '_').
                str.lower()
)
df = df.rename(columns={'d.o.b': 'date_of_birth', 'nationality(/ies)': 'nationality'},)

df = df.drop(
    columns=['name_non_latin_script','non_latin_script_type','non_latin_script_language',
             'other_information','passport_additional_information',
             # organisation details
             'type_of_entity','subsidiaries','parent_company','business_registration_number_(s)',
             # ship details
             'imo_number','current_owner/operator_(s)','previous_owner/operator_(s)',
             'current_believed_flag_of_ship','previous_flags','type_of_ship',
             'tonnage_of_ship','length_of_ship','year_built','hull_identification_number_(hin)'
])
df.drop_duplicates(inplace=True)

In [221]:
df['last_updated'] = pd.to_datetime(df['last_updated'], dayfirst=True)
df['date_designated'] = pd.to_datetime(df['date_designated'], dayfirst=True)

text_columns = df.select_dtypes(include='string').columns

for col in text_columns:
    df[col] = (
        df[col]
        .astype(str)
        .str.replace("\u2018", "'", regex=False) # open apostrophe
        .str.replace("\u2019", "'", regex=False) # closed apostrophe
        .str.replace("\u2013", "-", regex=False) # dash
        .str.replace(r'\s+', ' ', regex=True)
        .str.strip()
    )

# ensure formatting consistency
df['name_type'] = df['name_type'].str.title()
df['ofsi_group_id'] = df['ofsi_group_id'].astype('Int64')
df['gender'] = df['gender'].str.title()
df['town_of_birth'] = df['town_of_birth'].str.title()

# google to check standards of it
# df[['national_identifier_number', 'passport_number']] = (
#     df[['national_identifier_number', 'passport_number']]
#     .apply(lambda x: x.str.replace('[^a-zA-Z0-9]', '', regex=True)) 
# )

# keep pipe delimiter as is, ignore
# df['sanctions_imposed'] = df['sanctions_imposed'].str.replace('|', ', ').str.strip()

In [222]:
# name_1: first name, name_2-5: other/middle names, name_6: surname
name_cols = ['name_1', 'name_2', 'name_3', 'name_4', 'name_5', 'name_6']
address_col = ['address_line_1','address_line_2','address_line_3','address_line_4','address_line_5','address_line_6']

df['full_name'] = (df[name_cols]
    .apply(lambda x: x.str.title())
    .apply(lambda row: ' '.join(x for x in row if pd.notna(x) and x), axis=1)
)

df['address'] = (df[address_col]
    .apply(lambda row: ', '.join(x for x in row if pd.notna(x) and x), axis=1)
)

In [223]:
# def combine_text_columns(df, cols):
#     return (df[cols]
#             .apply(lambda x: x.str.strip().str.title())
#             .apply(lambda row: ' '.join(x for x in row if pd.notna(x) and x),
#             axis=1
#             )
#     )

# df['full_name'] = combine_text_columns(df, name_cols)
# df['address'] = combine_text_columns(df, address_col)

In [224]:
designation_drop = df[df['designation_type'].isin(['Entity', 'Ship'])].index


df.drop(designation_drop, inplace=True)

# not needed? 'title','alias_strength','un_reference_number', 'uk_statement_of_reasons', 
# 'designation_type', 'website',
# 'national_identifier_additional_information', 'designation_type'

df = df[['unique_id','ofsi_group_id',
         'full_name', 'name_type',
         'date_of_birth','nationality','gender','town_of_birth','country_of_birth',
         'phone_number','email_address',
         'national_identifier_number','passport_number',
         'address','address_postal_code','address_country',
         'regime_name','designation_source','date_designated','sanctions_imposed',
         'position','last_updated', ]]

df.to_csv('clean_individuals.csv', index=False)

In [225]:
# pd.options.display.max_seq_items = 1600

# # sorted(df['country_of_birth'].dropna().unique())
# # sorted(df['nationality'].dropna().unique())
# # sorted(df['address_country'].dropna().unique())
# # df['sanctions_imposed'].unique()
# df['address'].unique()

In [226]:
# primary_df = df[df['name_type'] == 'Primary Name'].copy()

primary_df = (df[df['name_type'] == 'Primary Name']
              .drop_duplicates(subset=['unique_id'])
              .copy()
)
# extract only full_name for aliases and group by unique_id

#3842

aliases_df = (
    df[df['name_type'] == 'Alias']
    .groupby('unique_id')['full_name']
    .apply(lambda x: '|'.join(sorted(set(x))))
    .reset_index()
    .rename(columns={'full_name': 'aliases'})
)

In [227]:
# primary_df
aliases_df

,unique_id,aliases
0,AFG0007,A Kabir
1,AFG0009,Mohammad Taher Anwari|Mohammad Tahre Anwari|Mu...
2,AFG0010,Sayyed Mohammad Haqqani
3,AFG0011,Abdul Latif Mansoor|Wali Mohammad
4,AFG0012,Shams Ur-Rahman Sher Alam|Shams-U-Rahman|Shams...
...,...,...
753,YEM0002,Abdallah Al-Hakim|Abdallah Al-Mu'Ayyad|Abu Ali...
754,YEM0004,Abdulmalik Al-Huthi
755,YEM0006,Abu Sagar Zabin|Abu Saqar Zabin
756,YEM0009,Abu Yasser


In [228]:
pd.options.display.max_seq_items = 1600

df['passport_number'].unique()

<StringArray>
[                                                               'P04581926',
                                                                 'D0009925',
                                                                        nan,
                                                                 'D0009720',
                                                                 'D0010442',
                                                                'SE-011697',
                                                                 'D0009726',
                                                                 'D0009751',
                                                                 'D0009803',
                                                                 'D0009716',
                                                                 'D0010098',
                                                                'D00009723',
                                                              

In [229]:
# def combine_unique(series):
#     values = series.dropna().astype(str)

#     cleaned = []
#     for v in values:
#         # parts = [x.strip() for x in v]#v.split(',')]
#         cleaned.extend(v)

#     # cleaned = [c for c in cleaned if c and c.lower() != 'nan']

#     return '|'.join(set((values)))

# combine_unique(pd.Series(['Khwaja Malik Village, Arghandab District, Kandahar Province', 'Kandahar City, Kandahar Province']))


In [230]:
def combine_unique(values):
    return '|'.join(sorted(set(v for v in values if pd.notna(v) and v)))

aggregated_df = df.groupby('unique_id').agg({
    'date_of_birth': combine_unique,
    'nationality': combine_unique,
    'town_of_birth': combine_unique,
    'passport_number': combine_unique,
    'national_identifier_number': combine_unique,
    'address': combine_unique,
    'address_country': combine_unique,
    # 'sanctions_imposed': combine_unique,
    'position': combine_unique
}).reset_index()

In [231]:
# final_df = (primary_df
#             .merge(aliases_df, on='unique_id', how='left')
#             .merge(aggregated_df, on='unique_id',how='left')
# )

# final_df = final_df.rename(columns={
#     'full_name': 'primary_name'
# })

# # final_df = final_df.drop_duplicates(subset=['unique_id'])

In [232]:
aggregate_cols = [
    'date_of_birth',
    'nationality',
    'town_of_birth',
    'passport_number',
    'national_identifier_number',
    'address',
    'address_country',
    'position'
]

primary_df = primary_df.drop(columns=aggregate_cols, errors='ignore')

final_df = (
    primary_df
    .merge(aliases_df, on='unique_id', how='left')
    .merge(aggregated_df, on='unique_id', how='left')
)

final_df = final_df.rename(columns={
    'full_name': 'primary_name'
})

final_df = final_df[
    ['unique_id','ofsi_group_id',
     'primary_name', 'aliases',
     'date_of_birth','nationality','gender','town_of_birth','country_of_birth',
     'phone_number','email_address',
     'national_identifier_number','passport_number',
     'address','address_postal_code','address_country',
     'regime_name','designation_source','date_designated','sanctions_imposed',
     'position','last_updated', ]
]

In [233]:
final_df.to_csv('../data/final.csv', index=False)